# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção da Infraestrutura da Linha de Paçoca

Neste notebook modelamos uma malha física de inspeção dos equipamentos e trechos de transporte
da planta de processamento de amendoim.

O objetivo é verificar se a malha permite um circuito Euleriano e, quando possível, utilizar o
Algoritmo de Hierholzer para gerar uma rota na qual o robô de inspeção percorra cada trecho da
malha uma única vez.

In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII."""
    if not dados:
        return "Tabela Vazia"

    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}

    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))

    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)

    linhas = [header, divisor]

    for row in dados:
        linhas.append(
            " | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas)
        )

    return "\n".join(linhas)


def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)

    header = f"{' ' * larg_linha} | " + " | ".join(
        f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols)
    )

    divisor = f"{'-' * larg_linha}-+-" + "-+-".join(
        "-" * larguras[j] for j in range(len(rotulos_cols))
    )

    linhas = [header, divisor]

    for i, r_nome in enumerate(rotulos_linhas):
        vals = []

        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")

        linhas.append(
            f"{r_nome:<{larg_linha}} | " + " | ".join(vals)
        )

    return "\n".join(linhas)


from collections import defaultdict
from typing import List, Tuple


class GrafoInspecaoEuleriano:
    def __init__(self):
        self.adj = defaultdict(list)

    def adicionar_trecho(self, u: str, v: str, id_trecho: str):
        # A malha de inspeção é não direcionada:
        # o robô pode percorrer o trecho nos dois sentidos.
        self.adj[u].append((v, id_trecho))
        self.adj[v].append((u, id_trecho))

    def verificar_euleriano(self) -> Tuple[bool, List[str]]:
        impares = [
            v for v, viz in self.adj.items()
            if len(viz) % 2 != 0
        ]

        return len(impares) == 0, impares

    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        adj_copia = {
            u: list(viz)
            for u, viz in self.adj.items()
        }

        pilha = [inicio]
        circuito = []

        while pilha:
            u = pilha[-1]

            if adj_copia[u]:
                v, id_trecho = adj_copia[u].pop()

                adj_copia[v].remove((u, id_trecho))
                pilha.append(v)

            else:
                circuito.append(pilha.pop())

        circuito.reverse()
        return circuito


# Malha de inspeção:
# cada equipamento possui conexão com dois trechos principais,
# e as ligações adicionais formam um circuito fechado.
g_insp = GrafoInspecaoEuleriano()

g_insp.adicionar_trecho("BASE_INSPECAO", "RECEPCAO", "I-01")
g_insp.adicionar_trecho("RECEPCAO", "LIMPEZA", "I-02")
g_insp.adicionar_trecho("LIMPEZA", "SECAGEM", "I-03")
g_insp.adicionar_trecho("SECAGEM", "SILO", "I-04")
g_insp.adicionar_trecho("SILO", "SELECAO_OPTICA", "I-05")
g_insp.adicionar_trecho("SELECAO_OPTICA", "TORRA", "I-06")
g_insp.adicionar_trecho("TORRA", "DESPEL.", "I-07")
g_insp.adicionar_trecho("DESPEL.", "MOAGEM", "I-08")
g_insp.adicionar_trecho("MOAGEM", "DOSAGEM", "I-09")
g_insp.adicionar_trecho("DOSAGEM", "PRENSA", "I-10")
g_insp.adicionar_trecho("PRENSA", "EMBALAGEM", "I-11")
g_insp.adicionar_trecho("EMBALAGEM", "BASE_INSPECAO", "I-12")

# Ramais internos que fecham a malha de inspeção.
g_insp.adicionar_trecho("SILO", "TORRA", "I-13")
g_insp.adicionar_trecho("TORRA", "MOAGEM", "I-14")
g_insp.adicionar_trecho("MOAGEM", "PRENSA", "I-15")
g_insp.adicionar_trecho("PRENSA", "SILO", "I-16")

eul, imp = g_insp.verificar_euleriano()

print(f"Grafo é Euleriano: {eul}")
print(f"Vértices Ímpares: {imp}")

rota_robo = g_insp.calcular_circuito_hierholzer("BASE_INSPECAO")

print("Circuito Euleriano de Inspeção:")
print(" -> ".join(rota_robo))

assert eul is True
assert len(rota_robo) == len(g_insp.adj) + 5
